# FrugalOptiX on RFMiD (Kaggle)
**Before you run:** right-side panel → *Session options* → Accelerator **GPU T4 x2 or P100**, Internet **On**.
Add two inputs: (1) the RFMiD dataset, (2) your `FrugalOptiX_v4.tar.gz` as a private dataset (Add Input → Upload).
Then *Run All*. Expect roughly 1-3 hours (lens design runs on CPU, training on GPU).

In [ ]:
import glob, os, subprocess, sys
tars = glob.glob('/kaggle/input/**/FrugalOptiX_v4.tar.gz', recursive=True)
assert tars, 'Upload FrugalOptiX_v4.tar.gz as a Kaggle dataset and add it as an input'
subprocess.run(['tar','-xzf',tars[0],'-C','/kaggle/working'],check=True)
os.chdir('/kaggle/working/FrugalOptiX'); print(os.getcwd())

In [ ]:
# torch/torchvision are already on Kaggle; install the rest
!pip install -q optiland==0.6.3 onnx onnxruntime scikit-image PyYAML
import torch; print('GPU:', torch.cuda.is_available())

In [ ]:
# Find the RFMiD dataset folder (any folder containing the *Label*.csv files)
import glob, os
labels = glob.glob('/kaggle/input/**/*[Ll]abel*.csv', recursive=True)
print(labels)
root = os.path.commonpath([os.path.dirname(p) for p in labels])
# go up to the dataset root so images in sibling folders are found
root = '/'.join(root.split('/')[:4])
print('root =', root)
!python scripts/merge_rfmid.py --root "{root}" --out data/RFMiD --max-side 768

In [ ]:
# THE RUN. SEEDS="0" is the fast option (about half the time of 3 seeds). Use SEEDS="0 1 2" for the final numbers.
!SEEDS="0" bash scripts/run_all.sh rfmid data/RFMiD --pretrained 2>&1 | grep -v -i "warning" | tail -60

In [ ]:
# Package everything you need to bring back: results + demo data + restorer weights
!cd /kaggle/working && zip -qr FrugalOptiX_results.zip FrugalOptiX/results FrugalOptiX/web FrugalOptiX/runs/restorer_selected.pt FrugalOptiX/runs/restorer.onnx
print('Download /kaggle/working/FrugalOptiX_results.zip from the Output tab')